# Step 4 — Pretrained Protein Sequence Model

## ESM-2 activity predictor

The baseline models use only simple mutation-level information.

The next experiment tests whether a pretrained protein language model
contains useful information about the local structural and evolutionary
context of a Cas9 residue.

We use the ESM-2 150M model:

`facebook/esm2_t30_150M_UR50D`

The ESM-2 encoder is frozen. Only the downstream prediction head is trained.

For each mutation, we use the ESM-2 representation corresponding to the
mutated residue position and combine it with the identity of the mutant
amino acid.

Because the Cas9 sequence has 1368 residues, while the standard ESM-2
extraction workflow uses a maximum sequence length of about 1022 residues,
we represent Cas9 using two overlapping sequence windows.

The primary evaluation remains the position-held-out split.

In [7]:
from pathlib import Path

# The notebook is stored inside the notebooks/ folder.
# Moving one level up gives us the project root.
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent

# Define the main project folders so all later cells use clean relative paths.
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

print("Current directory:")
print(CURRENT_DIR)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nData directory:")
print(DATA_DIR)

print("\nResults directory:")
print(RESULTS_DIR)

# Check that the important folders actually exist.
print("\nData directory exists:", DATA_DIR.exists())
print("Results directory exists:", RESULTS_DIR.exists())

Current directory:
c:\Users\subha\OneDrive\Desktop\Mandrake_Ai_Bio_research\Mandrake_Cas9_Assignment_Pack\Mandrake-Bio\notebooks

Project root:
c:\Users\subha\OneDrive\Desktop\Mandrake_Ai_Bio_research\Mandrake_Cas9_Assignment_Pack\Mandrake-Bio

Data directory:
c:\Users\subha\OneDrive\Desktop\Mandrake_Ai_Bio_research\Mandrake_Cas9_Assignment_Pack\Mandrake-Bio\data

Results directory:
c:\Users\subha\OneDrive\Desktop\Mandrake_Ai_Bio_research\Mandrake_Cas9_Assignment_Pack\Mandrake-Bio\results

Data directory exists: True
Results directory exists: True


In [8]:
import os
import random
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from transformers import AutoTokenizer, AutoModel

# Fix random seeds for reproducibility.
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use GPU when available because ESM-2 inference is much faster there.
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

Device: cpu


In [9]:
# Load the reference Cas9 sequence from the FASTA file.

WT_FASTA_PATH = DATA_DIR / "CAS9_STRP1_WT.fasta"

with open(WT_FASTA_PATH, "r") as f:
    fasta_lines = [line.strip() for line in f if line.strip()]

# The first line is the FASTA header; the remaining lines form the sequence.
wt_sequence = "".join(
    line for line in fasta_lines
    if not line.startswith(">")
)

print("WT sequence length:", len(wt_sequence))
print("First 30 residues:", wt_sequence[:30])
print("Last 30 residues:", wt_sequence[-30:])

WT sequence length: 1368
First 30 residues: MDKKYSIGLDIGTNSVGWAVITDEYKVPSK
Last 30 residues: TKEVLDATLIHQSITGLYETRIDLSQLGGD


In [10]:
# Facebook's 150M-parameter ESM-2 checkpoint.
# We use the frozen pretrained model only as a feature extractor.

ESM_MODEL_NAME = "facebook/esm2_t30_150M_UR50D"

tokenizer = AutoTokenizer.from_pretrained(
    ESM_MODEL_NAME
)

esm_model = AutoModel.from_pretrained(
    ESM_MODEL_NAME
)

# Move the model to GPU if available and disable training mode.
esm_model = esm_model.to(DEVICE)
esm_model.eval()

# We are not training ESM-2 in this experiment.
for parameter in esm_model.parameters():
    parameter.requires_grad = False

print("Loaded:", ESM_MODEL_NAME)
print("Device:", DEVICE)

C:\Users\subha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\subha\.cache\huggingface\hub\models--facebook--esm2_t30_150M_UR50D. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\subha\AppD

Loaded: facebook/esm2_t30_150M_UR50D
Device: cpu


In [11]:
# Read the hidden dimension directly from the model configuration.

ESM_HIDDEN_SIZE = esm_model.config.hidden_size

print("ESM hidden size:", ESM_HIDDEN_SIZE)

ESM hidden size: 640


In [12]:
# ESM-2's standard extraction workflow uses approximately 1022 residues.
# We therefore cover the 1368-residue Cas9 sequence using two overlapping windows.

MAX_WINDOW_LENGTH = 1022

# Window 1: N-terminal region
window_1_start = 0
window_1_end = min(
    window_1_start + MAX_WINDOW_LENGTH,
    len(wt_sequence)
)

# Window 2: C-terminal region.
# It overlaps substantially with Window 1 so that residues near the
# boundary still have useful local sequence context.
window_2_end = len(wt_sequence)
window_2_start = max(
    0,
    window_2_end - MAX_WINDOW_LENGTH
)

window_1 = wt_sequence[
    window_1_start:window_1_end
]

window_2 = wt_sequence[
    window_2_start:window_2_end
]

print("Window 1:", window_1_start + 1, "to", window_1_end)
print("Window 1 length:", len(window_1))

print("Window 2:", window_2_start + 1, "to", window_2_end)
print("Window 2 length:", len(window_2))

Window 1: 1 to 1022
Window 1 length: 1022
Window 2: 347 to 1368
Window 2 length: 1022


In [13]:
def extract_esm_embeddings(sequence):
    """
    Run the frozen ESM-2 model on one protein sequence.

    Returns:
        Tensor of shape [sequence_length, hidden_size]
    """

    # Tokenizer adds the special tokens required by ESM-2.
    inputs = tokenizer(
        sequence,
        return_tensors="pt"
    )

    # Move tokenized input to the same device as the model.
    inputs = {
        key: value.to(DEVICE)
        for key, value in inputs.items()
    }

    # No gradients are needed because ESM-2 is frozen.
    with torch.no_grad():
        outputs = esm_model(**inputs)

    # ESM output includes special tokens at the beginning/end.
    # Remove those tokens so that one row corresponds to each amino acid.
    embeddings = outputs.last_hidden_state[:, 1:-1, :]

    return embeddings.squeeze(0).cpu()


# Extract representations for the two Cas9 windows.
window_1_embeddings = extract_esm_embeddings(window_1)
window_2_embeddings = extract_esm_embeddings(window_2)

print("Window 1 embedding shape:",
      tuple(window_1_embeddings.shape))

print("Window 2 embedding shape:",
      tuple(window_2_embeddings.shape))

Window 1 embedding shape: (1022, 640)
Window 2 embedding shape: (1022, 640)


In [14]:
# Create one ESM representation for every Cas9 residue.
# This avoids running ESM-2 thousands of times for individual mutations.

cas9_embeddings = np.zeros(
    (len(wt_sequence), ESM_HIDDEN_SIZE),
    dtype=np.float32
)

for position in range(len(wt_sequence)):

    # Distance from the current residue to the edges of Window 1.
    if window_1_start <= position < window_1_end:

        local_position_1 = position - window_1_start

        distance_1 = min(
            local_position_1,
            len(window_1) - 1 - local_position_1
        )

    else:
        distance_1 = -1

    # Distance from the current residue to the edges of Window 2.
    if window_2_start <= position < window_2_end:

        local_position_2 = position - window_2_start

        distance_2 = min(
            local_position_2,
            len(window_2) - 1 - local_position_2
        )

    else:
        distance_2 = -1

    # Choose the window where the residue is farther from an edge.
    if distance_1 >= distance_2:

        local_position = position - window_1_start

        cas9_embeddings[position] = (
            window_1_embeddings[local_position].numpy()
        )

    else:

        local_position = position - window_2_start

        cas9_embeddings[position] = (
            window_2_embeddings[local_position].numpy()
        )


print("Cas9 embedding matrix shape:",
      cas9_embeddings.shape)

Cas9 embedding matrix shape: (1368, 640)


In [15]:
# Quick sanity check that every Cas9 position has a non-zero representation.

embedding_norms = np.linalg.norm(
    cas9_embeddings,
    axis=1
)

print("Minimum embedding norm:",
      embedding_norms.min())

print("Maximum embedding norm:",
      embedding_norms.max())

print("Number of residue embeddings:",
      len(embedding_norms))

Minimum embedding norm: 8.076697
Maximum embedding norm: 9.020917
Number of residue embeddings: 1368
